## **Executive Summary.**

### **Objective.**
Exploratory data analysis on real cryptocurrency derivatives market data (BTC and ETH perpetual futures) from Binance and Bybit exchanges, covering December 2025 - February 2026. The goal is to study the data, find patterns in trading activity, liquidity, and liquidations, and evaluate whether these patterns contain predictive information for building a filter signal.

### **Key Findings by Tier.**

**Foundational EDA (Data Quality + Basic Analysis).**
- Data is clean: zero missing values, no crossed spreads, no invalid values.
- Bybit generates ~2x more liquidation events than Binance, but total dollar volume is comparable. For cross-exchange comparisons, dollar volume should be used instead of event count (analogy with Dollar Bars by Lopez de Prado).
- Spreads are extremely tight (~0.01 bps for BTC).
- Liquidation volume is highly clustered: daily min/max ratio reaches 500x.

**Intermediate EDA (Feature Properties).**
- `mid_close` is non-stationary, other features exhibit stationarity in different ways, see explanation here: https://www.statsmodels.org/devel/examples/notebooks/generated/stationarity_detrending_adf_kpss.html.
- Returns show near-zero autocorrelation (market is efficient at the minute scale).
- Volatility features (mid_std, spread, trade_volume) are intercorrelated at 0.7–0.9. Liquidation features form a separate block (correlation with the volatility block is only 0.4–0.6), meaning liquidations carry unique information not duplicated by price volatility.
- All features exhibit heavy tails. Raw values break linear models - online EMA z-score (or any other analogues) normalization is required for use in training.
- Non-linearities are present: the relationship between features and future volatility is V-shaped (for returns) and concave (for volume). Tree-based models are appropriate.
- Three market regimes identified: calm (~50%), stressed (~45%), crisis (~5%). In crisis, volatility increases, spreads widen, and liquidation volume spikes sharply.

**Advanced EDA (Predictive Modeling + Validation).**
- Liquidation heatmaps confirm that liquidations cluster simultaneously in time and price, consistent with the cascading liquidation mechanism.
- The feature set for ML includes: recent volatility, trading volume, LOB spread-based liquidity proxy (motivated by the Slippage-at-Risk paper, Sepper 2026), liquidation volume/imbalance, and Amihud illiquidity coefficient (Lopez de Prado, Advances in Financial Machine Learning Book).
- XGBoost regression achieves positive $R^2$ on out-of-sample data when predicting liquidation volume over a 5-minute horizon. The model captures general stress regimes but does not predict individual spikes (expected for heavy-tailed data).
- XGBoost classifier for large liquidations (top 5%): BTC PR-AUC = 0.350, ETH PR-AUC = 0.274. At threshold 0.20, the BTC model flags 12.3% of minutes as high-risk and captures 47.5% of future large liquidation events.
- Monte Carlo stability analysis: PR-AUC standard deviation < 0.005 across 2000 random subsamples, the signal is stable and is not an artifact of a specific train/test split.
- Walk-forward validation: the predictive signal is present across all folds, but PR-AUC varies across time periods. The signal is regime-dependent - stronger during periods of liquidation clustering, weaker during calm periods.
- Calibration curves are close to the diagonal, meaning ensemble probabilities can be interpreted as approximate risk estimates, not just as rankings.
- Permutation importance confirms that the model is economically interpretable: trading volume and liquidation features dominate, spread-based liquidity features contribute, Amihud adds marginal value.

### **Practical Recommendations for Signal Construction.**
1. Liquidation features (volume, imbalance, rolling sums over 15–60 min horizons) are the strongest predictors. They should be the core of the filter signal.
2. Spread-based liquidity proxy (half-spread z-score) adds predictive power beyond volume - this is consistent with the SaR paper's finding that liquidity deterioration precedes cascades.
3. The model should be tree-based (XGBoost or analogues). Linear models are inappropriate due to non-linearities and heavy tails.
4. Features should be normalized online (EMA z-score) for working with a broader class of models.
5. The classifier output should be used as a continuous risk score (not a binary 0/1 decision). Threshold selection depends on the cost of false positives vs missed liquidations.
6. Walk-forward validation is essential: the signal weakens in calm regimes. Regime-dependent thresholds should be considered.

### **Limitations.**
- **No full order book depth.** Therefore, slippage and price impact metrics are approximate. With L2/L3 data, SaR-style analysis would be more precise.
- **Minute-level aggregation.** Sub-minute liquidation dynamics (which can unfold in seconds) are smoothed out. Higher-frequency features may improve recall.
- **Single asset class.** The analysis covers only BTC and ETH. Generalization to altcoins (less liquid and more volatile) has not been tested.
- **Three-month sample.** The period includes one major crash (February 2026) but may not represent all market conditions. A longer history would improve regime coverage.

### **References.**
- Lopez de Prado, M. (2018). *Advances in Financial Machine Learning.* Wiley.
- Sepper, O. (2026). *Slippage-at-Risk (SaR): A Forward-Looking Liquidity Risk Framework for Perpetual Futures Exchanges.* arXiv:2603.09164.